In [1]:
import json
import pandas as pd
from datetime import datetime


In [8]:
import zipfile



with zipfile.ZipFile('ai-club-dc-hackathon-25-26.zip') as z:   
    print("Files in ZIP:")
    z.printdir()          

Files in ZIP:
File Name                                             Modified             Size
ai-club-dc-hackathon-24-25/sample_submission.csv 2026-01-13 12:06:44        11021
ai-club-dc-hackathon-24-25/test_matchups.json  2026-01-13 12:06:44        16284
ai-club-dc-hackathon-24-25/train.json          2026-01-13 12:06:44        29516


In [10]:
with zipfile.ZipFile('ai-club-dc-hackathon-25-26.zip') as z:
    
    
    z.extract('ai-club-dc-hackathon-24-25/train.json')                    
    

In [11]:
zipfile.ZipFile('ai-club-dc-hackathon-25-26.zip').extract('ai-club-dc-hackathon-24-25/train.json')

'/drive/notebooks/ai-club-dc-hackathon-24-25/train.json'

In [211]:
print

<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [213]:
# Load train.json into memory
with open('/drive/notebooks/ai-club-dc-hackathon-24-25/train.json', 'r') as f:
    train_data = json.load(f)

print(type(train_data))
print(train_data.keys())  # seasons


<class 'dict'>
dict_keys(['2004-05', '2005-06', '2006-07', '2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17'])


In [214]:
rows = []
# Iterate through seasons and knockout rounds in the training data
for season, rounds in train_data.items():
    for round_name, matches in rounds.items():
        for match in matches:

            # Normalize team names to ensure consistency across datasets
            team_1 = normalize_team(match["team_1"])
            team_2 = normalize_team(match["team_2"])
            winner = normalize_team(match["winner"])

             # Convert match date to datetime format for temporal consistency
            date = pd.to_datetime(match["date"], dayfirst=True)

         # Store a single match as one row
            rows.append({
                "season": season,
                "round": round_name,
                "team_1": team_1,
                "team_2": team_2,
                "winner": winner,
                "date": date
            })

# Create a structured DataFrame from parsed match records
train_df = pd.DataFrame(rows)
# Preview the processed training data
train_df.head()


,season,round,team_1,team_2,winner,date
0,2004-05,round_of_16,Real Madrid,Juventus,Juventus,2005-02-22
1,2004-05,round_of_16,Liverpool,Bayer Leverkusen,Liverpool,2005-02-22
2,2004-05,round_of_16,PSV Eindhoven,Monaco,PSV Eindhoven,2005-02-22
3,2004-05,round_of_16,Bayern Munich,Arsenal,Bayern Munich,2005-02-22
4,2004-05,round_of_16,Barcelona,Chelsea,Chelsea,2005-02-23


In [19]:
print(train_df.shape)
print(train_df["round"].value_counts())
print(train_df["season"].nunique())


(195, 6)
round
round_of_16       104
quarter_finals     52
semi_finals        26
final              13
Name: count, dtype: int64
13


In [82]:
# 1. Remove matches where both teams are same
train_df = train_df[train_df["team_1"] != train_df["team_2"]]

# 2. Remove rows where winner is invalid
train_df = train_df[
    (train_df["winner"] == train_df["team_1"]) |
    (train_df["winner"] == train_df["team_2"])
]

# 3. Drop any remaining missing values
train_df = train_df.dropna().reset_index(drop=True)

print("After cleaning:", train_df.shape)


After cleaning: (192, 8)


In [83]:
train_df = train_df.sort_values("date").reset_index(drop=True)

print("Date range:")
print(train_df["date"].min(), "→", train_df["date"].max())


Date range:
2005-02-22 00:00:00 → 2017-04-12 00:00:00


In [223]:
train_df["target"] = (train_df["winner"] == train_df["team_1"]).astype(int)

train_df[["team_1", "team_2", "winner", "target"]].head()


,team_1,team_2,winner,target
0,Real Madrid,Juventus,Juventus,0
1,Liverpool,Bayer Leverkusen,Liverpool,1
2,PSV Eindhoven,Monaco,PSV Eindhoven,1
3,Bayern Munich,Arsenal,Bayern Munich,1
4,Barcelona,Chelsea,Chelsea,0


In [234]:
# Map knockout rounds to ordinal stage values
# Higher stages correspond to more important matches

ROUND_MAP = {
    "round_of_16": 1,
    "quarter_finals": 2,
    "semi_finals": 3,
    "final": 4
}
# Convert round names into numerical stage values
train_df["stage"] = train_df["round"].map(ROUND_MAP)

# Verify distribution of matches across stages
train_df["stage"].value_counts().sort_index()


stage
1    104
2     52
3     24
4     12
Name: count, dtype: int64

In [235]:
CUTOFF_DATE = pd.to_datetime("2017-04-30")

train_df = train_df[train_df["date"] <= CUTOFF_DATE].reset_index(drop=True)

print("Max date after cutoff:", train_df["date"].max())


Max date after cutoff: 2017-04-12 00:00:00


In [233]:
print("Final train_df shape:", train_df.shape)
print("Seasons:", train_df["season"].nunique())
print("Rounds:\n", train_df["round"].value_counts())

assert train_df["date"].max() <= CUTOFF_DATE
assert train_df["target"].isin([0,1]).all()

print("STEP 2 COMPLETED SUCCESSFULLY ")


Final train_df shape: (192, 8)
Seasons: 13
Rounds:
 round
round_of_16       104
quarter_finals     52
semi_finals        24
final              12
Name: count, dtype: int64
STEP 2 COMPLETED SUCCESSFULLY 


In [227]:
full_df = pd.read_csv('/drive/notebooks/Full_Dataset.csv')

# make columns lowercase 
full_df.columns = full_df.columns.str.lower()

# parse date
full_df["date"] = pd.to_datetime(full_df["date"], errors="coerce")

# cutoff 
CUTOFF_DATE = pd.to_datetime("2017-04-30")
full_df = full_df[full_df["date"] <= CUTOFF_DATE]

# normalize team names
full_df["team"] = full_df["team"].apply(normalize_team)
full_df["opponent"] = full_df["opponent"].apply(normalize_team)

full_df.head()


<ipython-input-227-32d121c8de1b>:7: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  full_df["date"] = pd.to_datetime(full_df["date"], errors="coerce")


,round,date,time,team,team_score,opponent_score,opponent,home_score_aet,away_score_aet,home_penalties,away_penalties,team_points,opponent_points,season,location,country,competition
0,ROUND 1,2002-08-31,22:30,RACING SANTANDER,0.0,1.0,VALLADOLID,NaN,NaN,NaN,NaN,0.0,3.0,2002,Home,spain,primera-division
1,ROUND 1,2002-09-01,21:00,RAYO VALLECANO,2.0,2.0,ALAVES,NaN,NaN,NaN,NaN,1.0,1.0,2002,Home,spain,primera-division
2,ROUND 1,2002-09-01,21:00,REAL SOCIEDAD,4.0,2.0,ATHLETIC BILBAO,NaN,NaN,NaN,NaN,3.0,0.0,2002,Home,spain,primera-division
3,ROUND 1,2002-09-01,22:00,MALLORCA,0.0,2.0,VALENCIA,NaN,NaN,NaN,NaN,0.0,3.0,2002,Home,spain,primera-division
4,ROUND 1,2002-09-01,22:30,VILLARREAL,2.0,2.0,OSASUNA,NaN,NaN,NaN,NaN,1.0,1.0,2002,Home,spain,primera-division


In [228]:
# Replace 'column_name' with  actual column
full_df['team'].unique()

# Better: with count
full_df['team'].value_counts()

team
BARCELONA               825
REAL MADRID             802
CHELSEA                 782
MANCHESTER UTD          781
ARSENAL                 776
                       ... 
SV WALDKIRCH              1
ALEMANNIA WALDALGES.      1
SCHOTT JENA               1
TSG PFEDDERSHEIM          1
MATERA                    1
Name: count, Length: 1315, dtype: int64

In [236]:
#  Only domestic league matches
domestic_df = full_df[
    ~full_df["competition"].str.contains("champions|uefa|europa", case=False, na=False)
].copy()

print(domestic_df["competition"].value_counts().head())


competition
fa-cup              13482
ligue-1             11338
primera-division    11336
premier-league      11326
serie-a             11024
Name: count, dtype: int64


In [230]:
domestic_df["win"] = (domestic_df["team_score"] > domestic_df["opponent_score"]).astype(int)
domestic_df["goal_diff"] = domestic_df["team_score"] - domestic_df["opponent_score"]


In [132]:

# Aggregate domestic league statistics at team level
domestic_features = domestic_df.groupby("team").agg(
    domestic_matches=("win", "count"),
    domestic_wins=("win", "sum"),
    avg_goal_diff=("goal_diff", "mean")
).reset_index()

# Compute domestic win rate as a stable measure of baseline team strength
domestic_features["domestic_win_rate"] = (
    domestic_features["domestic_wins"] /
    domestic_features["domestic_matches"]
)

domestic_features.head()


,team,domestic_matches,domestic_wins,avg_goal_diff,domestic_win_rate
0,1899 HOFFENHEIM,341,122,0.090909,0.357771
1,AC MILAN,610,331,0.703279,0.542623
2,ACCRINGTON ST,31,11,-0.064516,0.354839
3,ACIREALE,3,0,-1.666667,0.000000
4,ACR MESSINA,126,28,-0.500000,0.222222


In [237]:
ucl_rows = []

for _, row in train_df.iterrows():
    # team_1 frame
    ucl_rows.append({
        "team": row["team_1"],
        "won": 1 if row["target"] == 1 else 0,
        "stage": row["stage"]
    })

    # team_2 frame
    ucl_rows.append({
        "team": row["team_2"],
        "won": 1 if row["target"] == 0 else 0,
        "stage": row["stage"]
    })

ucl_long = pd.DataFrame(ucl_rows)
ucl_long.head(10)


,team,won,stage
0,Real Madrid,0,1
1,Juventus,1,1
2,Liverpool,1,1
3,Bayer Leverkusen,0,1
4,PSV Eindhoven,1,1
5,Monaco,0,1
6,Bayern Munich,1,1
7,Arsenal,0,1
8,Barcelona,0,1
9,Chelsea,1,1


In [135]:
# Aggregate UCL performance metrics across seasons
ucl_features = ucl_long.groupby("team").agg(
    ucl_matches=("won", "count"),
    ucl_wins=("won", "sum"),
    avg_stage=("stage", "mean")
).reset_index()

# Compute UCL win rate to capture continental performance

ucl_features["ucl_win_rate"] = (
    ucl_features["ucl_wins"] / ucl_features["ucl_matches"]
)

ucl_features.head()


,team,ucl_matches,ucl_wins,avg_stage,ucl_win_rate
0,AC Milan,18,10,1.888889,0.555556
1,APOEL,2,1,1.500000,0.500000
2,Ajax,1,0,1.000000,0.000000
3,Arsenal,20,7,1.550000,0.350000
4,Atletico Madrid,13,9,2.076923,0.692308


In [138]:
# Assign higher weights to later knockout stages
# This captures tournament pedigree and high-pressure experience

STAGE_WEIGHTS = {1: 1, 2: 3, 3: 6, 4: 10}
# Convert stage values into weighted scores
ucl_long["stage_score"] = ucl_long["stage"].map(STAGE_WEIGHTS)

# Aggregate pedigree-related features at team level
pedigree = ucl_long.groupby("team").agg(
    pedigree_score=("stage_score", "sum"),
    finals_played=("stage", lambda x: (x == 4).sum())
).reset_index()


In [139]:
# Combine UCL performance, pedigree, and domestic league features
team_features = (
    ucl_features
    .merge(pedigree, on="team", how="left")
    .merge(domestic_features, on="team", how="left")
)
# Replace missing values for teams with incomplete history
team_features = team_features.fillna(0)
team_features.head()


,team,ucl_matches,ucl_wins,avg_stage,ucl_win_rate,pedigree_score,finals_played,domestic_matches,domestic_wins,avg_goal_diff,domestic_win_rate
0,AC Milan,18,10,1.888889,0.555556,59,2,0.0,0.0,0.0,0.0
1,APOEL,2,1,1.500000,0.500000,4,0,0.0,0.0,0.0,0.0
2,Ajax,1,0,1.000000,0.000000,1,0,0.0,0.0,0.0,0.0
3,Arsenal,20,7,1.550000,0.350000,47,1,0.0,0.0,0.0,0.0
4,Atletico Madrid,13,9,2.076923,0.692308,49,2,0.0,0.0,0.0,0.0


In [140]:
print(team_features.columns)
print(team_features.shape)


Index(['team', 'ucl_matches', 'ucl_wins', 'avg_stage', 'ucl_win_rate',
       'pedigree_score', 'finals_played', 'domestic_matches', 'domestic_wins',
       'avg_goal_diff', 'domestic_win_rate'],
      dtype='object')
(52, 11)


In [152]:
# Merge team_1 features
train_feat = train_df.merge(
    team_features,
    left_on="team_1",
    right_on="team",
    how="left"
)

# Rename team_1 feature columns
train_feat = train_feat.rename(columns={
    col: f"{col}_t1"
    for col in team_features.columns
    if col != "team"
})

# Merge team_2 features
train_feat = train_feat.merge(
    team_features,
    left_on="team_2",
    right_on="team",
    how="left"
)

# Rename team_2 feature columns
train_feat = train_feat.rename(columns={
    col: f"{col}_t2"
    for col in team_features.columns
    if col != "team"
})


In [153]:
print(sorted(train_feat.columns))


['avg_goal_diff_t1', 'avg_goal_diff_t2', 'avg_stage_t1', 'avg_stage_t2', 'date', 'domestic_matches_t1', 'domestic_matches_t2', 'domestic_win_rate_t1', 'domestic_win_rate_t2', 'domestic_wins_t1', 'domestic_wins_t2', 'finals_played_t1', 'finals_played_t2', 'pedigree_score_t1', 'pedigree_score_t2', 'round', 'season', 'stage', 'target', 'team_1', 'team_2', 'team_x', 'team_y', 'ucl_matches_t1', 'ucl_matches_t2', 'ucl_win_rate_t1', 'ucl_win_rate_t2', 'ucl_wins_t1', 'ucl_wins_t2', 'winner']


In [156]:
# Build match-level feature differences
# This allows the model to learn relative team strength instead of absolute values

X = pd.DataFrame({
    "ucl_win_rate_diff": train_feat["ucl_win_rate_t1"] - train_feat["ucl_win_rate_t2"],
    "ucl_matches_diff": train_feat["ucl_matches_t1"] - train_feat["ucl_matches_t2"],
    "pedigree_diff": train_feat["pedigree_score_t1"] - train_feat["pedigree_score_t2"],
    "domestic_win_rate_diff": train_feat["domestic_win_rate_t1"] - train_feat["domestic_win_rate_t2"],
    "avg_goal_diff_diff": train_feat["avg_goal_diff_t1"] - train_feat["avg_goal_diff_t2"],
    "stage": train_feat["stage"]
})
# Binary target: whether team_1 won the match
y = train_feat["target"]

X.head()


,ucl_win_rate_diff,ucl_matches_diff,pedigree_diff,domestic_win_rate_diff,avg_goal_diff_diff,stage
0,0.071429,14,52,0.0,0.0,1
1,0.714286,9,50,0.0,0.0,1
2,-0.171429,2,7,0.0,0.0,1
3,0.327419,11,61,0.0,0.0,1
4,0.096154,10,46,0.0,0.0,1


In [157]:
from sklearn.model_selection import train_test_split
# Split data while preserving win/loss distribution

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
# Confirm split dimensions
print(X_train.shape, X_val.shape)


(153, 6) (39, 6)


In [158]:
from sklearn.preprocessing import StandardScaler
# Standardize features to zero mean and unit variance
scaler = StandardScaler()
# Fit scaler only on training data to avoid leakage
X_train_s = scaler.fit_transform(X_train)
# Apply same transformation to validation data
X_val_s = scaler.transform(X_val)


In [159]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Logistic Regression: stable baseline with interpretable coefficients
log_model = LogisticRegression(max_iter=1000, random_state=42)

# Random Forest: captures non-linear interactions and reduces variance
rf_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=7,
    min_samples_leaf=8,
    random_state=42
)
# Gradient Boosting: sequentially corrects errors for complex patterns
gb_model = GradientBoostingClassifier(
    n_estimators=250,
    learning_rate=0.04,
    max_depth=3,
    random_state=42
)
# Train models
log_model.fit(X_train_s, y_train)
rf_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)


,loss,'log_loss'
,learning_rate,0.04
,n_estimators,250
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [160]:
from sklearn.metrics import accuracy_score, roc_auc_score

# Collect validation probabilities for comparison
models = {
    "Logistic": log_model.predict_proba(X_val_s)[:, 1],
    "RandomForest": rf_model.predict_proba(X_val)[:, 1],
    "GradientBoosting": gb_model.predict_proba(X_val)[:, 1]
}
# Evaluate each model using ROC–AUC and accuracy
for name, probs in models.items():
    print(
        name,
        "AUC:", roc_auc_score(y_val, probs),
        "ACC:", accuracy_score(y_val, (probs >= 0.5))
    )


Logistic AUC: 0.8428571428571429 ACC: 0.7948717948717948
RandomForest AUC: 0.8285714285714286 ACC: 0.7435897435897436
GradientBoosting AUC: 0.7714285714285715 ACC: 0.7692307692307693


In [161]:
# Weights chosen to balance stability and ranking performance
W_LOG, W_RF, W_GB = 0.3, 0.3, 0.4

# Individual model probabilities
p_log = log_model.predict_proba(X_val_s)[:, 1]
p_rf  = rf_model.predict_proba(X_val)[:, 1]
p_gb  = gb_model.predict_proba(X_val)[:, 1]

# Weighted soft-voting ensemble
ensemble_p = W_LOG*p_log + W_RF*p_rf + W_GB*p_gb


In [257]:
ensemble_probs = ensemble_p

print(
    "Ensemble AUC:",
    roc_auc_score(y_val, ensemble_probs),
    "Ensemble ACC:",
    accuracy_score(y_val, (ensemble_probs >= 0.5))
)


Ensemble AUC: 0.8028571428571429 Ensemble ACC: 0.7692307692307693


In [239]:
import joblib

joblib.dump(log_model, "log_model.pkl")
joblib.dump(rf_model, "rf_model.pkl")
joblib.dump(gb_model, "gb_model.pkl")
joblib.dump(platt, "platt.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(team_features, "team_features.pkl")


['team_features.pkl']

In [240]:
import zipfile




with zipfile.ZipFile('ai-club-dc-hackathon-25-26.zip') as z:  
    print("Files in ZIP:")
    z.printdir()          
    

Files in ZIP:
File Name                                             Modified             Size
ai-club-dc-hackathon-24-25/sample_submission.csv 2026-01-13 12:06:44        11021
ai-club-dc-hackathon-24-25/test_matchups.json  2026-01-13 12:06:44        16284
ai-club-dc-hackathon-24-25/train.json          2026-01-13 12:06:44        29516


In [241]:
with zipfile.ZipFile('ai-club-dc-hackathon-25-26.zip') as z:
    
    
    z.extract('ai-club-dc-hackathon-24-25/test_matchups.json')                   

In [242]:
zipfile.ZipFile('ai-club-dc-hackathon-25-26.zip').extract('ai-club-dc-hackathon-24-25/test_matchups.json')

'/drive/notebooks/ai-club-dc-hackathon-24-25/test_matchups.json'

In [243]:
team_features = joblib.load("team_features.pkl")  # Load precomputed team-level features for test-time inference

In [244]:
import zipfile



with zipfile.ZipFile('ai-club-dc-hackathon-25-26.zip') as z:  
    print("Files in ZIP:")
    z.printdir()          

Files in ZIP:
File Name                                             Modified             Size
ai-club-dc-hackathon-24-25/sample_submission.csv 2026-01-13 12:06:44        11021
ai-club-dc-hackathon-24-25/test_matchups.json  2026-01-13 12:06:44        16284
ai-club-dc-hackathon-24-25/train.json          2026-01-13 12:06:44        29516


In [245]:
with zipfile.ZipFile('ai-club-dc-hackathon-25-26.zip') as z:
    
    
    z.extract('ai-club-dc-hackathon-24-25/sample_submission.csv')                    
    
    

In [246]:
zipfile.ZipFile('ai-club-dc-hackathon-25-26.zip').extract('ai-club-dc-hackathon-24-25/sample_submission.csv')

'/drive/notebooks/ai-club-dc-hackathon-24-25/sample_submission.csv'

In [247]:
print("Current winner keys:", list(winners.keys())[:10])


Current winner keys: ['Winner of RB Leipzig vs Manchester City', 'Winner of R161', 'Winner of Club Brugge vs Benfica', 'Winner of R162', 'Winner of Liverpool vs Real Madrid', 'Winner of R163', 'Winner of AC Milan vs Tottenham Hotspur', 'Winner of R164', 'Winner of Eintracht Frankfurt vs Napoli', 'Winner of R165']


In [254]:
import pandas as pd

sample = pd.read_csv('/drive/notebooks/ai-club-dc-hackathon-24-25/sample_submission.csv')
print(sample.head())


   id   season                                        predictions
0   0  2017-18  {"round_of_16": [{"team_1": "Real Madrid", "te...
1   1  2018-19  {"round_of_16": [{"team_1": "Barcelona", "team...
2   2  2019-20  {"round_of_16": [{"team_1": "Real Madrid", "te...
3   3  2020-21  {"round_of_16": [{"team_1": "Barcelona", "team...
4   4  2021-22  {"round_of_16": [{"team_1": "Real Madrid", "te...


In [255]:
final_submission = sample.copy()
final_submission[["predicted_winner", "win_probability"]] = pd.DataFrame(rows)

final_submission.head()


<class 'ValueError'>: Columns must be same length as key

In [258]:
final_submission.to_csv("final_submission.csv", index=False)
print("final_submission.csv generated ")


final_submission.csv generated 
